Notebook pre-requisites:

In [608]:
!pip install "camelot-py[cv]" > /dev/null 2>&1
!pip install spacy > /dev/null 2>&1


In [609]:
!pip install PyMuPDF  > /dev/null 2>&1
!pip install pdfplumber  > /dev/null 2>&1

## 1. PDF Ingestion & Parsing
- Extract text with page/section anchors (page number, heading hierarchy).
- Preserve structure: titles, subsections, lists, tables, figures’ captions.
- For tables: parse into machine-readable frames (CSV/JSON) when possible.
- Deliverables: raw_text.jsonl (chunks with metadata), tables/*.csv.

In [610]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="camelot")

In [611]:
import fitz
import json
import re
from pathlib import Path
import shutil
import camelot
import pdfplumber
import pandas as pd

# =============== TOC EXTRACTION ===============
def extract_contents_section(doc):
    toc_lines = []
    toc_started = False
    current_title = ""
    suppressing = False


    # regular expressions
    toc_header_re = re.compile(r"\b(Contents|Table of Contents)\b", re.IGNORECASE)
    toc_entry_re = re.compile(r"\.{3,}\s*(\d+)$")
    thl_re = re.compile(r"THL", re.IGNORECASE)
    food_re = re.compile(r"FOOD", re.IGNORECASE)

    # iterate through pages
    # extract TOC lines
    # stop when no TOC entries found in last few lines
    for page in doc:
        lines = page.get_text().splitlines()
        for line in lines:
            stripped = line.strip()
            if not stripped:
                continue
            if not toc_started and toc_header_re.search(stripped):
                toc_started = True
                continue
            if not toc_started:
                continue

            # suppression logic
            if suppressing:
                m_food = food_re.search(stripped)
                if m_food:
                    stripped = stripped[m_food.end():].strip()
                    suppressing = False
                    if not stripped:
                        continue
                else:
                    continue

            m_thl = thl_re.search(stripped)
            if m_thl:
                m_food = food_re.search(stripped, m_thl.end())
                if m_food:
                    stripped = stripped[m_food.end():].strip()
                    if not stripped:
                        continue
                else:
                    stripped = stripped[:m_thl.start()].strip()
                    suppressing = True
                    if not stripped:
                        continue

            match = toc_entry_re.search(stripped)
            if match:
                full_title = (current_title + " " + stripped).strip() if current_title else stripped
                toc_lines.append(full_title)
                current_title = ""
            else:
                if current_title:
                    current_title += " " + stripped
                else:
                    current_title = stripped

        last_few = lines[-5:]
        if toc_started and all(not toc_entry_re.search(l.strip()) for l in last_few):
            break

    if current_title:
        if suppressing:
            last_thl = re.search(r"THL", current_title, re.IGNORECASE)
            cleaned_title = current_title[:last_thl.start()].strip() if last_thl else ""
        else:
            cleaned_title = current_title.strip()
        if cleaned_title:
            toc_lines.append(cleaned_title)

    return "\n".join(toc_lines)

# =============== PARSE CONTENTS TO DATAFRAME ===============
def parse_contents_to_df(contents_text):
    lines = [l.strip() for l in contents_text.split("\n") if re.search(r"\d+\s*$", l)]
    rows = []
    for line in lines:
        match = re.match(r"(.+?)\s+(\d+)$", line)
        if match:
            title, page = match.groups()
            title = re.sub(r"\.{2,}", "", title).strip()
            rows.append([title, int(page)])
    df = pd.DataFrame(rows, columns=["title", "page"])
    df = merge_split_rows(df)
    return df

def merge_split_rows(df):
    return df

def normalize_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text.strip().lower()

# =============== TABLE EXTRACTION USING CAMELT/PLUMBER ===============
def extract_tables(pdf_path, output_dir):
    output_dir = Path(output_dir)
    tables_dir = output_dir / "tables"
    if tables_dir.exists():
        shutil.rmtree(tables_dir)
    tables_dir.mkdir(exist_ok=True)

    all_tables = []
    doc = fitz.open(pdf_path)

    for page_num in range(len(doc)):
        page_str = str(page_num + 1)
        page_tables = []

        # Camelot STREAM --> extract tables using stream flavor
        try:
            tables_stream = camelot.read_pdf(pdf_path, pages=page_str, flavor='stream', edge_tol=50, row_tol=10, strip_text='\n')
            page_tables += [t.df for t in tables_stream if not t.df.empty]
        except Exception:
            pass

        # Camelot LATTICE --> extract tables using lattice flavor
        if not page_tables:
            try:
                tables_lattice = camelot.read_pdf(pdf_path, pages=page_str, flavor='lattice', line_scale=40, shift_text=['l','t'])
                page_tables += [t.df for t in tables_lattice if not t.df.empty]
            except Exception:
                pass

        # pdfplumber fallback
        if not page_tables:
            with pdfplumber.open(pdf_path) as pdf:
                page = pdf.pages[page_num]
                plumber_tables = page.extract_tables()
                for pt in plumber_tables:
                    df = pd.DataFrame(pt)
                    page_tables.append(df)

        # save tables in the data/tables/ directory in CSV format
        for i, df in enumerate(page_tables):
            df = df.fillna("").astype(str).apply(lambda col: col.map(lambda x: re.sub(r"\n", " ", x).strip()))
            df = merge_split_rows(df)
            table_file = tables_dir / f"table_page{page_num+1}_{i+1}.csv"
            df.to_csv(table_file, index=False)
            all_tables.append({
                "table_number": i + 1,
                "page": page_num + 1,
                "file": str(table_file),
                "rows": df.shape[0],
                "columns": df.shape[1]
            })

    print(f"✓ Extracted {len(all_tables)} tables total to '{tables_dir}'")
    return all_tables

# =============== MAIN EXTRACTION FUNCTION ===============
def extract_pdf_with_structure(pdf_path, output_dir="data"):
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    doc = fitz.open(pdf_path)
    chunks = []

    # --- Extract TOC and parse ---
    contents_text = extract_contents_section(doc)
    toc_df = parse_contents_to_df(contents_text)
    toc_df = toc_df.sort_values(by="page").reset_index(drop=True)

    toc_index = 0
    current_section = ""

    for page_num in range(len(doc)):
        page = doc[page_num]
        lines = page.get_text().splitlines()

        new_sections = []
        while toc_index < len(toc_df) and toc_df.loc[toc_index, "page"] <= page_num + 1:
            new_sections.append(toc_df.loc[toc_index, "title"])
            toc_index += 1

        pending_sections = new_sections.copy()
        buffer = []
        line_idx = 0

        while line_idx < len(lines):
            matched_section = None
            matched_lines_count = 0
            for section_title in pending_sections:
                for lookahead in range(1, min(5, len(lines) - line_idx + 1)):
                    candidate_text = " ".join(lines[line_idx:line_idx+lookahead])
                    if normalize_text(candidate_text).startswith(normalize_text(section_title)):
                        matched_section = section_title
                        matched_lines_count = lookahead
                        break
                if matched_section:
                    break

            if matched_section:
                if current_section and buffer:
                    text_chunk = " ".join(buffer).strip()
                    if text_chunk:
                        chunks.append({"page": page_num+1, "section": current_section, "text": text_chunk})
                current_section = matched_section
                buffer = []
                pending_sections.remove(matched_section)
                line_idx += matched_lines_count
                continue
            else:
                if current_section:
                    buffer.append(lines[line_idx].strip())
                line_idx += 1

        if current_section and buffer:
            text_chunk = " ".join(buffer).strip()
            if text_chunk:
                chunks.append({"page": page_num+1, "section": current_section, "text": text_chunk})

        if pending_sections:
            for missing_section in pending_sections:
                print(f"[WARNING] Section '{missing_section}' expected on page {page_num+1} but not found.")
                if buffer:
                    text_chunk = " ".join(buffer).strip()
                    if text_chunk:
                        chunks.append({"page": page_num+1, "section": missing_section, "text": text_chunk})
                        buffer = []

    # save text in JSONL format
    output_file = output_dir / "raw_text.jsonl"
    with open(output_file, "w", encoding="utf-8") as f:
        for chunk in chunks:
            f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

    print(f"Extracted {len(chunks)} text chunks")
    print(f"Clean text saved to {output_file}")

    # extract tables
    tables_info = extract_tables(pdf_path, output_dir)

    return chunks, toc_df, tables_info


In [612]:
pdf_path = "data/sustainable-health-from-food_web.pdf"
chunks, toc_df, tables = extract_pdf_with_structure(pdf_path)

[WARNING] Section 'Appendix 8. Recommended intakes of fat, carbohydrates and protein for adults and children over 2 years (without alcohol and with fibre taken into account)' expected on page 102 but not found.
Extracted 134 text chunks
Clean text saved to data/raw_text.jsonl
✓ Extracted 137 tables total to 'data/tables'


## 3. NER and Keyphrase extraction

NER and keyphrase extraction:
- Run a domain-adapted NER pipeline (spaCy/transformer) + keyphrase extraction (YAKE/KeyBERT) over text chunks.
- Map mentions to ontology classes (entity typing).
- Resolve coreference (merge aliases, e.g., “vitamin C” ↔ “ascorbic acid”).
- Deliverable: entities.jsonl with canonical IDs and mention spans.

Deliverable contains one JSON object per canonical entity: each entity contains a stable ID, a canonical label, type (one of the ontology classes), synonyms/aliases, and all mention occurrences with text spans + provenance (page, section, char offsets).

Inputs are:
- raw_text.jsonl and tables/*.csv
- ontology.yaml

**Step 1**: create `seed_vocabularies.json`

The file contains examples of entities extracted from the text.

How it works:
- read the extracted text `raw_text.jsonl` on chunk at a time and clean it
- look for mentions of ontology classes using keyword and regex lists (for each chunk count terms only once)
- after scanning the whole text, keep only the terms that appeared in at least 2 different chunks 
- save the terms with relative ontology class in `seed_vocabularies.json`


In [613]:
import unicodedata
from collections import defaultdict, Counter

RAW = Path("data/raw_text.jsonl")

def norm(s: str) -> str:
    # Unicode fold + lowercase + collapse whitespace; keep hyphens so we can match variants
    s = unicodedata.normalize("NFKD", s).casefold().strip()
    s = re.sub(r"\s+", " ", s)
    return s

# Helpers to build robust patterns 
def esc_variants(term: str) -> str:
    """
    Build a regex that matches normalized text variants:
    - spaces -> [\\s-]+ (space or hyphen)
    - keep digits, allow hyphen/space around digits in common constructs (omega-3, type 2)
    Assumes we'll search in casefolded, normalized text.
    """
    t = term.strip().casefold()
    # protect regex metachars by escaping first
    t = re.escape(t)
    # allow space/hyphen variants
    t = t.replace(r"\ ", r"[\s\-]+")
    # special: omega-3 / omega 3; type 2 diabetes / type-2-diabetes already covered by space->class
    return t

def compile_list(terms, word_boundaries=True):
    alts = [esc_variants(t) for t in terms]
    if not alts:
        return None
    alt = "(?:" + "|".join(sorted(set(alts), key=len, reverse=True)) + ")"
    if word_boundaries:
        return re.compile(rf"\b{alt}\b", re.I)
    return re.compile(alt, re.I)

# Existing domain patterns
vitamin_rx = re.compile(r"\bvitamin[s]?\s+[a-z](?:\d+)?\b", re.I)  # search on normalized text
minerals_list = ["calcium","iron","zinc","iodine","selenium","potassium","magnesium","phosphorus"]
macro_micro_list = [
    "fibre","fiber","saturated fat","unsaturated fat","omega-3","omega-6",
    "protein","carbohydrate","sodium","salt","sugars","free sugar","added sugar"
]
food_groups = [
    "cereals","grains","whole grain","vegetables","berries","fruits","legumes","pulses",
    "nuts","seeds","fish","seafood","red meat","processed meat","poultry","milk","dairy",
    "eggs","fats","oils","beverages","alcohol","potatoes","rye","oats","barley"
]
tech_terms = [
    "smoking","salting","nitrite","curing","fermentation","pasteurisation","pasteurization",
    "boiling","frying","baking","grilling","fortification","iodised salt","iodized salt"
]
guideline_verbs = ["limit","reduce","increase","prefer","choose","eat more","aim to","replace","avoid"]

health_outcome_terms = [
    "cardiovascular disease","ischemic heart disease","stroke","type 2 diabetes","obesity","overweight",
    "hypertension","high blood pressure","blood pressure","cholesterol","ldl cholesterol","hdl cholesterol",
    "triglycerides","cancer","colorectal cancer","all-cause mortality","mortality","morbidity","inflammation",
    "insulin resistance","metabolic syndrome"
]
environment_terms = [
    "greenhouse gas emissions","ghg emissions","carbon footprint","climate impact","environmental impact",
    "land use","water use","water footprint","nitrogen footprint","phosphorus footprint","biodiversity",
    "biodiversity loss","ecological footprint","emissions","food system emissions","sustainability"
]

risk_intro = r"(?:risk|risk of|associated with|linked to|higher odds of|increases|reduces)"

# Precompiled alternations against normalized text
minerals_rx   = compile_list(minerals_list)
macromicro_rx = compile_list(macro_micro_list)
foods_rx      = compile_list(food_groups)
tech_rx       = compile_list(tech_terms)
health_rx     = compile_list(health_outcome_terms)
env_rx        = compile_list(environment_terms)

# heads we accept after risk-intros (guard against vague tails)
risk_heads = compile_list([
    "cardiovascular disease","ischemic heart disease","type 2 diabetes","obesity","overweight",
    "hypertension","high blood pressure","blood pressure","ldl cholesterol","hdl cholesterol",
    "triglycerides","cancer","colorectal cancer","mortality","all-cause mortality","stroke",
    "cholesterol","inflammation"
])

deny_tails = re.compile(r"\b(of|to|in|for|on|with|by|at|from)$")

def per_chunk_increment(counter: Counter, key: str, seen: set):
    if key not in seen:
        counter[key] += 1
        seen.add(key)

gaz = defaultdict(Counter)
provenance = defaultdict(lambda: defaultdict(list))  # provenance[category][term] -> list of {"chunk","section","page","excerpt"}

def add_prov(cat, term, chunk_id, section, page, text_norm, match_span, max_len=160):
    start, end = match_span
    s = max(0, start - 60)
    e = min(len(text_norm), end + 60)
    snippet = text_norm[s:e]
    if len(provenance[cat][term]) < 3:  # cap examples to keep file small
        provenance[cat][term].append({
            "chunk": chunk_id, "section": section, "page": page,
            "excerpt": snippet
        })

with open(RAW, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        section = obj.get("section") or ""
        text = obj.get("text","")
        page = obj.get("page")
        cid = obj.get("id") or obj.get("chunk_id") or f"chunk_{page}"

        # Skip likely non-content sections 
        section_raw = (obj.get("section") or obj.get("heading") or "").lower()
        if any(s in section_raw for s in ["table of contents","contents","acknowledgement","acknowledgment","reference","bibliography"]):
            continue

        section_n = norm(section)
        text_n = norm(text)

        # Per-chunk seen sets for doc-frequency counting
        seen_nutr, seen_ing, seen_tech, seen_guid, seen_health, seen_env = [set() for _ in range(6)]

        # Nutrients 
        for m in vitamin_rx.finditer(text_n):
            per_chunk_increment(gaz["nutrient"], m.group(0), seen_nutr)
            add_prov("nutrient", m.group(0), cid, section_n, page, text_n, m.span())

        for rx in [minerals_rx, macromicro_rx]:
            if not rx: continue
            for m in rx.finditer(text_n):
                term = m.group(0)
                per_chunk_increment(gaz["nutrient"], term, seen_nutr)
                add_prov("nutrient", term, cid, section_n, page, text_n, m.span())

        # Ingredients 
        if foods_rx:
            for m in foods_rx.finditer(text_n):
                term = m.group(0)
                per_chunk_increment(gaz["ingredient"], term, seen_ing)
                add_prov("ingredient", term, cid, section_n, page, text_n, m.span())

        # Techniques
        if tech_rx:
            for m in tech_rx.finditer(text_n):
                term = m.group(0)
                per_chunk_increment(gaz["technique"], term, seen_tech)
                add_prov("technique", term, cid, section_n, page, text_n, m.span())

        # Guidelines 
        for v in guideline_verbs:
            # require boundary after verb; capture up to 6 tokens tail
            for m in re.finditer(rf"\b{re.escape(v)}\s+[a-z][a-z\- ]{{2,60}}", text_n, re.I):
                phrase = m.group(0).strip()
                phrase = re.sub(r"\s+", " ", phrase)
                if not deny_tails.search(phrase.split()[-1]):
                    per_chunk_increment(gaz["guideline"], phrase, seen_guid)
                    add_prov("guideline", phrase, cid, section_n, page, text_n, m.span())

        # Health Outcomes
        if health_rx:
            for m in health_rx.finditer(text_n):
                term = m.group(0)
                per_chunk_increment(gaz["healthOutcome"], term, seen_health)
                add_prov("healthOutcome", term, cid, section_n, page, text_n, m.span())

        # Risk-intro patterns: “… reduces blood pressure”, “… increases LDL cholesterol”
        risk_pat = re.compile(rf"\b{risk_intro}\s+([a-z][a-z\- ]{{2,60}}?)\b", re.I)
        for m in risk_pat.finditer(text_n):
            tail = m.group(1).strip()
            tail = re.sub(r"\s+", " ", tail)
            if deny_tails.search(tail.split()[-1]):
                continue
            # Accept only if tail contains one of our outcome heads (precompiled)
            if risk_heads and risk_heads.search(tail):
                per_chunk_increment(gaz["healthOutcome"], tail, seen_health)
                add_prov("healthOutcome", tail, cid, section_n, page, text_n, m.span(1))

        # Environmental Impacts 
        if env_rx:
            for m in env_rx.finditer(text_n):
                term = m.group(0)
                per_chunk_increment(gaz["environmentImpact"], term, seen_env)
                add_prov("environmentImpact", term, cid, section_n, page, text_n, m.span())

        # common abbrev
        if re.search(r"\bghg\b", text_n):
            per_chunk_increment(gaz["environmentImpact"], "ghg emissions", seen_env)
            # provenance without exact span: approximate
            i = text_n.find("ghg")
            add_prov("environmentImpact", "ghg emissions", cid, section_n, page, text_n, (i, i+3))

# Keep items seen >=2 chunks (document frequency)
THRESHOLD = 2
seed_vocab = {k: sorted([term for term, c in cnts.items() if c >= THRESHOLD]) for k, cnts in gaz.items()}

print("Seeds summary (doc-frequency):", {k: len(v) for k, v in seed_vocab.items()})
for k, terms in seed_vocab.items():
    print(f"\n{k.upper()} ({len(terms)}):")
    for t in terms[:60]:
        print(" •", t)

# Write outputs
with open("data/seed_vocabularies.json", "w", encoding="utf-8") as out:
    json.dump(seed_vocab, out, ensure_ascii=False, indent=2)

print("\n Saved seed_vocabularies.json")


Seeds summary (doc-frequency): {'nutrient': 26, 'ingredient': 25, 'guideline': 2, 'environmentImpact': 10, 'technique': 6, 'healthOutcome': 15}

NUTRIENT (26):
 • added sugar
 • calcium
 • carbohydrate
 • fiber
 • fibre
 • free sugar
 • iodine
 • iron
 • magnesium
 • phosphorus
 • potassium
 • protein
 • salt
 • saturated fat
 • selenium
 • sugars
 • unsaturated fat
 • vitamin a
 • vitamin b12
 • vitamin b6
 • vitamin c
 • vitamin d
 • vitamin d3
 • vitamin e
 • vitamin k
 • zinc

INGREDIENT (25):
 • alcohol
 • barley
 • berries
 • beverages
 • cereals
 • dairy
 • eggs
 • fats
 • fish
 • fruits
 • grains
 • legumes
 • milk
 • nuts
 • oats
 • oils
 • potatoes
 • poultry
 • processed meat
 • red meat
 • rye
 • seafood
 • seeds
 • vegetables
 • whole grain

GUIDELINE (2):
 • aim to influence the population
 • increase blood cholesterol levels

ENVIRONMENTIMPACT (10):
 • biodiversity
 • biodiversity loss
 • carbon footprint
 • climate impact
 • emissions
 • environmental impact
 • greenhou

**Step 2**: create `new_seed.json`

The file contains examples of entities extracted from the text and revised. Starting from `seed_vocabularies.json`, eliminate usless examples and add relevant missing ones.


**Step 3**: create `entities_new.jsonl`

The file contains unique entities found in the text and keeps track of ID and mentions.

How it works:
- load inputs: raw text, seeds and ontology
- use spaCy to find mentions of seeds in the text 
- merge alias and create entities by picking the most generic name as label, generating ID (with ontology prefix) and collecting mentions
- save one line per each unique entity in `entities_new.jsonl`


In [614]:
%pip install -q spacy /dev/null 2>&1


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
ERROR: Invalid requirement: '/dev/null': Expected package name at the start of dependency specifier
    /dev/null
    ^
Hint: It looks like a path. The path does exist. The argument you provided (/dev/null) appears to be a requirements file. If that is the case, use the '-r' flag to install the packages specified within it.
Note: you may need to restart the kernel to use updated packages.


In [615]:
import hashlib
import yaml
import spacy
from spacy.matcher import PhraseMatcher
from spacy.util import filter_spans
from spacy.tokens import Span

RAW_PATH   = Path("data/raw_text.jsonl")
SEEDS_PATH = Path("data/new_seed.json")
ONTO_PATH  = Path("data/ontology.yaml")
OUT_PATH   = Path("data/entities_new.jsonl")

# Ontology essentials 
onto = yaml.safe_load(ONTO_PATH.read_text(encoding="utf-8")) or {}
BASE_URI = (
    onto.get("meta", {}).get("base_uri")
    or onto.get("base_uri")
    or "http://example.org/food#"
)
PREFIX = onto.get("prefixes", {}).get("ex", BASE_URI)

print("Base URI:", BASE_URI)
print("Prefix  :", PREFIX)

# Allowed classes 
VALID_CLASSES = {
    "nutrient": "nutrient",
    "ingredient": "ingredient",
    "technique": "technique",
    "dietaryGuideline": "dietaryGuideline",
    "healthOutcome": "healthOutcome",
    "environmentImpact": "environmentImpact",
}

# Map seed groups -> ontology classes 
LABEL_MAP = {
    "nutrient": "nutrient",
    "ingredient": "ingredient",
    "technique": "technique",
    "guideline": "dietaryGuideline",
    "healthOutcome": "healthOutcome",
    "environmentImpact": "environmentImpact",
}

# Load seeds
seeds = json.loads(SEEDS_PATH.read_text(encoding="utf-8"))
for k in list(seeds.keys()):
    if k not in LABEL_MAP:
        print("Skipping unknown seed group:", k)

print("Loaded seeds:", {k: len(v) for k, v in seeds.items() if k in LABEL_MAP})

# Normalization helpers
def norm_text(s: str) -> str:
    s = unicodedata.normalize("NFKD", s).casefold().strip()
    s = re.sub(r"\s+", " ", s)
    return s

def alias_norm(s: str) -> str:
    # Aggressive normalization for alias matching
    s = unicodedata.normalize("NFKD", s).casefold()
    s = s.replace("-", " ")
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def safe_slug(s: str) -> str:
    base = alias_norm(s)
    base = re.sub(r"[^a-z0-9]+", "_", base).strip("_")
    if not base:
        base = hashlib.md5(s.encode("utf-8")).hexdigest()[:8]
    return base

def context_window_sentence(doc, start_char):
    # Sentence text containing the span; fallback to None
    for sent in doc.sents:
        if sent.start_char <= start_char < sent.end_char:
            return sent.text
    return None

# spaCy pipeline: blank English + sentencizer + PhraseMatcher 
nlp = spacy.blank("en")
if "sentencizer" not in nlp.pipe_names:
    nlp.add_pipe("sentencizer")

matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

def add_with_variants(label: str, term: str):
    term = term.strip()
    if not term:
        return
    docs = [nlp.make_doc(term)]
    # add common hyphen/space variants
    if "-" in term:
        docs.append(nlp.make_doc(term.replace("-", " ")))
    if " " in term:
        docs.append(nlp.make_doc(term.replace(" ", "-")))
    # de-dup variants
    _seen = set()
    uniq_docs = []
    for d in docs:
        key = d.text.lower()
        if key not in _seen:
            _seen.add(key)
            uniq_docs.append(d)
    matcher.add(label, uniq_docs)

added = 0
for group, terms in seeds.items():
    label = LABEL_MAP.get(group)
    if not label:
        continue
    for term in terms:
        add_with_variants(label, term)
        added += 1

print(f"PhraseMatcher loaded with {added} seed terms across {len([k for k in seeds if k in LABEL_MAP])} groups.")

# Section filtering (skip non-content sections)
NONCONTENT = (
    "table of contents", "contents", "acknowledgement", "acknowledgment",
    "reference", "references", "bibliography", "appendix"
)

# Collect mentions with overlap resolution and per-chunk dedup
mentions = []

with RAW_PATH.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        obj = json.loads(line)
        text = obj.get("text", "")
        section = obj.get("section", "") or obj.get("heading", "") or ""
        if any(s in section.lower() for s in NONCONTENT):
            continue

        doc = nlp(text)

        # Gather spans from matcher
        spans = []
        for match_id, start, end in matcher(doc):
            label = nlp.vocab.strings[match_id]  # already ontology class name
            # Only accept labels defined in VALID_CLASSES (normalized compare)
            if label not in VALID_CLASSES.values():
                continue
            spans.append(Span(doc, start, end, label=label))

        # Resolve overlaps (keep best/longest)
        spans = filter_spans(spans)

        # Per-chunk dedup: identical (type, start, end) once
        seen = set()
        for sp in spans:
            key = (sp.label_, sp.start_char, sp.end_char)
            if key in seen:
                continue
            seen.add(key)

            mentions.append({
                "chunk_id": obj.get("id") or obj.get("chunk_id") or f"raw:{i:06d}",
                "page": obj.get("page"),
                "section": section,
                "type": sp.label_,
                "text_span": [int(sp.start_char), int(sp.end_char)],
                "surface": sp.text,
                "context": context_window_sentence(doc, sp.start_char),
            })

print(f"Collected {len(mentions)} mentions.")
for x in mentions[:5]:
    print(x)

# Alias/coref merge (union-find) + protected pairs
ALIASES = {
    "ascorbic acid": ["vitamin c"],
    "vitamin c": ["ascorbic acid"],
    "folate": ["folic acid","vitamin b9"],
    "cobalamin": ["vitamin b12"],
    "retinol": ["vitamin a"],
    "omega-3": ["n-3","omega 3","n3"],
    "omega-6": ["n-6","omega 6","n6"],
    "polyunsaturated fatty acids": ["pufa","polyunsaturated fat","polyunsaturated fats"],
    "monounsaturated fatty acids": ["mufa","monounsaturated fat","monounsaturated fats"],
    "saturated fatty acids": ["sfa","saturated fat","saturated fats"],
}

DO_NOT_MERGE = {
    tuple(sorted(("added sugar","free sugar"))),
    tuple(sorted(("red meat","processed meat"))),
    tuple(sorted(("ldl cholesterol","hdl cholesterol"))),
    tuple(sorted(("land use","water use"))),
}

class UnionFind:
    def __init__(self):
        self.parent = {}
    def find(self, x):
        self.parent.setdefault(x, x)
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    def union(self, a, b):
        pa, pb = self.find(a), self.find(b)
        if pa != pb:
            self.parent[pb] = pa

def build_alias_unionfind():
    uf = UnionFind()
    for k, vs in ALIASES.items():
        ak = alias_norm(k)
        for v in vs:
            uf.union(ak, alias_norm(v))
    return uf

uf = build_alias_unionfind()

# Merge mentions into canonical entities (per type), emit stable IDs
by_type = defaultdict(list)
for m in mentions:
    by_type[m["type"]].append(m)

entities = {}

def make_id(label, etype):
    # ID includes type prefix for collision safety
    return f"{PREFIX}{etype}_{safe_slug(label)}"

for etype, mlist in by_type.items():
    # bucket by alias representative
    buckets = defaultdict(list)
    for m in mlist:
        key = alias_norm(m["surface"])
        rep = uf.find(key)
        # protect pairs that should not merge
        if tuple(sorted((key, rep))) in DO_NOT_MERGE:
            rep = key
        buckets[(etype, rep)].append(m)

    # build canonical entity per bucket
    for (etype, rep), ms in buckets.items():
        # choose most frequent surface form (case-preserved) as label
        cap_counts = Counter([m["surface"] for m in ms])
        label = max(cap_counts, key=cap_counts.get)
        eid = make_id(label, etype)

        if eid not in entities:
            entities[eid] = {
                "id": eid,
                "type": etype,
                "label": label,
                "aliases": sorted({alias_norm(m["surface"]) for m in ms
                                   if alias_norm(m["surface"]) != alias_norm(label)}),
                "source": "sustainable-health-from-food.pdf",
                "mentions": [],
            }
        entities[eid]["mentions"].extend(ms)

# finalize: order mentions and write JSONL
for e in entities.values():
    e["mentions"].sort(key=lambda m: ((m.get("page") or 0), m["text_span"][0]))

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUT_PATH.open("w", encoding="utf-8") as out:
    for e in entities.values():
        out.write(json.dumps(e, ensure_ascii=False) + "\n")

print(f"Wrote {len(entities)} canonical entities -> {OUT_PATH}")


Base URI: http://example.org/food#
Prefix  : http://example.org/food#
Loaded seeds: {'nutrient': 35, 'ingredient': 43, 'guideline': 4, 'environmentImpact': 13, 'technique': 11, 'healthOutcome': 22}
PhraseMatcher loaded with 128 seed terms across 6 groups.
Collected 1537 mentions.
{'chunk_id': 'raw:000001', 'page': 5, 'section': 'Preface', 'type': 'ingredient', 'text_span': [556, 560], 'surface': 'salt', 'context': 'Finnish food habits have improved in recent decades, but the challenges that remain are excessive salt and saturated fat intakes and insufficient fibre intake.'}
{'chunk_id': 'raw:000001', 'page': 5, 'section': 'Preface', 'type': 'nutrient', 'text_span': [575, 578], 'surface': 'fat', 'context': 'Finnish food habits have improved in recent decades, but the challenges that remain are excessive salt and saturated fat intakes and insufficient fibre intake.'}
{'chunk_id': 'raw:000001', 'page': 5, 'section': 'Preface', 'type': 'dietaryGuideline', 'text_span': [651, 659], 'surface'

## 4. Relation Extraction & Triple Building
- Use rule-based patterns and/or relation extraction models to detect relations (e.g., Ingredient X contains
Nutrient Y, Technique Z requires Temperature T).
- Convert to triples (RDF or property graph). Include provenance (page, line span).
- Deduplicate and validate (schema consistency).
- Deliverables: triples.ttl (RDF) or graph.json (property graph).

In [616]:
!pip install --upgrade nltk > /dev/null 2>&1

In [617]:

from rdflib import Graph, Namespace
from rdflib.namespace import RDFS

# simple sentence tokenizer
def simple_sent_tokenize(text):
    if not text:
        return []
    # split on sentence ending punctuation followed by whitespace and uppercase letter
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return sentences




#INPUT_FILE = "data/entities_lowercased.jsonl"
INPUT_FILE = "data/entities_new.jsonl"
OUTPUT_FILE = "data/triples_clean.ttl"
EX = Namespace("http://example.org/food#")


# load all mentions and preserve every occurrence
# by also keeping memory of the sections in which they were found
mentions = []
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        entity = json.loads(line)
        for m in entity.get("mentions", []):
            mentions.append({
                "entity": entity["label"].strip().lower(),
                "ontology_type": entity["type"],
                "page": m.get("page"),
                "section": m.get("section", "Unknown"),
                "context": m.get("context", ""),
                "surface": m.get("surface", "")
            })

print(f"Loaded {len(mentions)} mentions from {INPUT_FILE}")


# group mentions by (page, section)
# so if a page has three sections --> three separate groups instead of one 
# to avoid having relations extracted across different sections of the same page
mentions_by_group = defaultdict(list)
for m in mentions:
    key = (m["page"], m["section"])
    mentions_by_group[key].append(m)

print(f"Grouped into {len(mentions_by_group)} (page, section) groups")


# define keywords for each relation type
RELATION_KEYWORDS = {
    "hasNutrient": ["contain", "source of", "rich in", "provides", "includes"],
    "usesTechnique": ["cooked", "prepared with", "using", "baked", "boiled"],
    "hasGuideline": ["recommended", "advised", "guideline", "suggested"],
    "associatedWithOutcome": ["linked to", "associated with", "causes", "reduces", "increases"],
    "hasEnvironmentalImpact": ["emission", "carbon", "sustainability", "environmental impact"],
    "affectsRiskOf": ["reduces risk", "increases risk", "affects", "lowers chance"],
    "recommendsTechnique": ["recommend", "advise", "encourage use of"],
    "aimsToImprove": ["aims to", "intended to", "improve", "enhance"],
    "guidelineTargetsImpact": ["reduce", "minimize", "targets impact", "sustainable"],
    "affectsImpactCategory": ["impact", "pollution", "emission"]
}

# helper functions
# if two mentions are in the same sentence retrurn True
def in_same_sentence(m1, m2):
    context = m1.get("context", "")
    if not context:
        return False, None

    sentences = simple_sent_tokenize(context)
    for sent in sentences:
        if m1["surface"] in sent and m2["surface"] in sent:
            return True, sent
    return False, None

# check if any keyword is in the context
def has_trigger(context, keywords):
    context_lower = context.lower()
    return any(kw in context_lower for kw in keywords)



# define relation extraction rules
def extract_relations(group_entities):
    triples = []

    # for each pair of entities in the group
    for e1 in group_entities:
        pred = None
        for e2 in group_entities:
            # skip same entity mentions
            if e1["entity"] == e2["entity"]:
                continue

            # only consider sentences in e1's context
            sentences = simple_sent_tokenize(e1.get("context", ""))
            # give section info
            found_in_same_sentence = False
            for sent in sentences:
                if e1["surface"].lower() in sent.lower() and e2["surface"].lower() in sent.lower():
                    # debug print what is found 
                    # print also page and section
                    # print(f"Section: {e1.get('section', 'Unknown')} found match, page: {e1.get('page', 'Unknown')}")
                    # print("---"
                    #      f"Entity 1: {e1['surface']}, Entity 2: {e2['surface']}")
                    # print the sentence in which they were both found
                    # print(f"Found in same sentence: {sent}")
                    found_in_same_sentence = True
                    break
            
            # if both entities are in the same sentence
            # proceed to check for relation triggers for creation of triples
            # otherwise skip
            if found_in_same_sentence:
                # print for debug
                # print(f"Debug: Considering entities {e1['entity']} and {e2['entity']} found in same sentence.")
                # Determine predicate based on ontology types
                t1, t2 = e1["ontology_type"], e2["ontology_type"]
                if t1 == "ingredient" and t2 == "nutrient":
                    pred = "hasNutrient"
                elif t1 == "ingredient" and t2 == "technique":
                    pred = "usesTechnique"
                elif t1 == "ingredient" and t2 == "dietaryGuideline":
                    pred = "hasGuideline"
                elif t1 == "ingredient" and t2 == "healthOutcome":
                    pred = "associatedWithOutcome"
                elif t1 == "ingredient" and t2 == "environmentImpact":
                    pred = "hasEnvironmentalImpact"
                elif t1 == "nutrient" and t2 == "healthOutcome":
                    pred = "affectsRiskOf"
                elif t1 == "dietaryGuideline" and t2 == "technique":
                    pred = "recommendsTechnique"
                elif t1 == "dietaryGuideline" and t2 == "healthOutcome":
                    pred = "aimsToImprove"
                elif t1 == "dietaryGuideline" and t2 == "environmentImpact":
                    pred = "guidelineTargetsImpact"
                elif t1 == "technique" and t2 == "environmentImpact":
                    pred = "affectsImpactCategory"
                
                if pred:
                    # append triple found
                    triples.append((e1["entity"], pred, e2["entity"]))

    return list(set(triples))


# build RDF graph
g = Graph()
g.bind("ex", EX)


# extract triples with provenance comments
triples_with_comments = []
triple_count = 0

# process each group separately
# to avoid cross-section/page relations
for (page, section), group_entities in mentions_by_group.items():
    relations = extract_relations(group_entities)
    # for each relation found, create triple with comment
    # including page and section info
    for subj, pred, obj in relations:
        subj_uri = f"ex:{subj.replace(' ', '_')}"
        pred_uri = f"ex:{pred.replace(' ', '_')}"
        obj_uri = f"ex:{obj.replace(' ', '_')}"
        comment = f"Found on page {page}, section '{section}'"
        triples_with_comments.append((subj_uri, pred_uri, obj_uri, comment))
        triple_count += 1


# save results to the output TTL file 
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write("@prefix ex: <http://example.org/food#> .\n")
    f.write("@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\n\n")

    for subj, pred, obj, comment in triples_with_comments:
        f.write(f"{subj} {pred} {obj} ;\n")
        f.write(f'    rdfs:comment "{comment}" .\n\n')

print(f"Added {triple_count} triples (each with a comment) → {OUTPUT_FILE}")


Loaded 1537 mentions from data/entities_new.jsonl
Grouped into 94 (page, section) groups
Added 486 triples (each with a comment) → data/triples_clean.ttl


## Point 5

- Load triples into a graph store (RDF triplestore like GraphDB/Fuseki, or Neo4j).
- Create indices; verify counts by class/relation.
- Run sample SPARQL or Cypher queries to sanity-check coverage.
- Deliverable: query notebook with 5-10 example queries --> notebook with Fuseki queries can be found in `data/fuseki_queries.ipynb`

In [618]:
!pip install rdflib pandas > /dev/null 2>&1

In [619]:
# Load your triples.ttl file
g = Graph()
g.parse("data/triples_clean.ttl", format="turtle")

# find how many triples excluding comments are found
EX = Namespace("http://example.org/food#")
triples = set()
for s, p, o in g:
    if p != RDFS.comment:
        triples.add((s, p, o))
        # print(f"Found triple: {s}, {p}, {o}")

print(f"Loaded {len(triples)} triples")


Loaded 289 triples


In [620]:
EX = Namespace("http://example.org/ex#")
g.bind("ex", EX)

In [621]:
# Q1: count total triples of facts found (excluding comments)

q1 = """
SELECT (COUNT(*) AS ?factTriples)
WHERE {
  ?s ?p ?o .
  FILTER(STRSTARTS(STR(?p), "http://example.org/food#"))
}
"""

for row in g.query(q1):
    print("Total Triples:", row.factTriples)


Total Triples: 289


In [622]:
# Q2: list all entities that are linked to nutrients, limit to first 20

q2 = """
SELECT ?entity ?nutrient
WHERE {
  ?entity ex:hasNutrient ?nutrient .
}
LIMIT 20
"""
pd.DataFrame(g.query(q2), columns=["Entity", "Nutrient"])


,Entity,Nutrient
0,http://example.org/food#fish,http://example.org/food#fat
1,http://example.org/food#nuts,http://example.org/food#fat
2,http://example.org/food#salt,http://example.org/food#fat
3,http://example.org/food#vegetable_oils,http://example.org/food#fat
4,http://example.org/food#dairy,http://example.org/food#fat
5,http://example.org/food#berries,http://example.org/food#fat
6,http://example.org/food#vegetables,http://example.org/food#fat
7,http://example.org/food#fruits,http://example.org/food#fat
8,http://example.org/food#legumes,http://example.org/food#fat
9,http://example.org/food#potatoes,http://example.org/food#fat


In [623]:
# Q3: extract all comments associated with entities, order by comment text, limit to first 5

q3 = """
SELECT ?entity ?comment
WHERE {
  ?entity rdfs:comment ?comment .
}
ORDER BY ?comment
LIMIT 5
"""

results = g.query(q3)
df = pd.DataFrame(results, columns=["Entity", "Comment"])

# extract page number from comment
df["Page"] = df["Comment"].str.extract(r"(\d+)")
df.drop(columns=["Comment"], inplace=True)
df


,Entity,Page
0,http://example.org/food#processed_meat,10
1,http://example.org/food#butter,11
2,http://example.org/food#oils,11
3,http://example.org/food#cheese,11
4,http://example.org/food#fish,11


## Point 6
- Materialize atomic facts from the KG (subject-predicate-object) with natural-language paraphrases.
- Detect intra-document contradictions or overlaps (e.g., numeric conflicts across pages); flag for curation.
- Deliverables: facts.jsonl with fields: {triple, text_support, page, paraphrases[]}.


In [624]:
# load triples.ttl file
g = Graph()
g.parse("data/triples_clean.ttl", format="turtle")
EX = Namespace("http://example.org/ex#")
g.bind("ex", EX)


In [625]:
# print sample predicates from TTL
print("Sample predicates from TTL:")
for p in set(g.predicates()):
    print(p)


Sample predicates from TTL:
http://www.w3.org/2000/01/rdf-schema#comment
http://example.org/food#recommendsTechnique
http://example.org/food#hasEnvironmentalImpact
http://example.org/food#aimsToImprove
http://example.org/food#guidelineTargetsImpact
http://example.org/food#usesTechnique
http://example.org/food#hasGuideline
http://example.org/food#affectsRiskOf
http://example.org/food#associatedWithOutcome
http://example.org/food#hasNutrient


In [626]:
# load triples with comments
EX = Namespace("http://example.org/food#")

# function to extract triples along with their comments
# extracting comments for each triple is crucial to keep track of the source page
def extract_facts_with_comments(ttl_path):
    with open(ttl_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    triple_comment_pairs = []
    current_triple = None
    current_comment = None

    # parse the raw TTL text manually to associate each comment with its triple
    for line in lines:
        line = line.strip()

        # match triple lines like: ex:legumes ex:associatedWithOutcome ex:mortality
        triple_match = re.match(r'^(ex:\w+)\s+(ex:\w+)\s+(ex:\w+)', line)
        if triple_match:
            current_triple = triple_match.groups()

        # match rdfs:comment "Found on page XX"
        comment_match = re.search(r'rdfs:comment\s+"([^"]+)"', line)
        if comment_match and current_triple:
            current_comment = comment_match.group(1)
            triple_comment_pairs.append((current_triple, current_comment))
            current_triple = None  # reset for next triple

    # load RDF into rdflib graph to confirm valid URIs
    g = Graph()
    g.parse(ttl_path, format="turtle")

    # build final structured list of facts
    facts = []
    for (s_prefix, p_prefix, o_prefix), comment in triple_comment_pairs:
        s = s_prefix.split(":")[1]
        p = p_prefix.split(":")[1]
        o = o_prefix.split(":")[1]

        # extract page number from comment and save it
        match = re.search(r'page (\d+)', comment)
        page = match.group(1) if match else None

        facts.append({
            "subject": s,
            "predicate": p,
            "object": o,
            "page": page
        })

    return facts


# function to create paraphrases based on predicate type
# this is useful for generating natural language variations of the facts
# these paraphrases can be used for various NLP tasks
def make_paraphrases(row):
    s, p, o = row["subject"], row["predicate"], row["object"]

    templates = {
        "hasNutrient": [
            f"{s} contains {o}.",
            f"{o} is a nutrient found in {s}."
        ],
        "usesTechnique": [
            f"{s} is prepared using {o}.",
            f"The preparation of {s} involves {o}."
        ],
        "hasGuideline": [
            f"{s} follows the guideline: {o}.",
            f"The dietary guideline {o} applies to {s}."
        ],
        "recommendsTechnique": [
            f"The guideline {s} recommends {o}.",
            f"{s} suggests using the technique {o}."
        ],
        "aimsToImprove": [
            f"The guideline {s} aims to improve {o}.",
            f"{s} targets the health outcome {o}."
        ],
        "affectsRiskOf": [
            f"{s} affects the risk of {o}.",
            f"Consuming {s} is associated with risk of {o}."
        ],
        "associatedWithOutcome": [
            f"{s} is associated with the health outcome {o}.",
            f"{s} has a relationship with {o}."
        ],
        "hasEnvironmentalImpact": [
            f"{s} has an environmental impact: {o}.",
            f"{s} contributes to {o} impact."
        ],
        "guidelineTargetsImpact": [
            f"The guideline {s} targets environmental impact {o}.",
            f"{s} aims to reduce or manage {o}."
        ],
        "affectsImpactCategory": [
            f"{s} affects the environmental impact category {o}.",
            f"The technique {s} tends to influence {o}."
        ],
    }

    return templates.get(p, [f"{s} {p} {o}."])



# extract facts with comments using our triples_clean ttl file
facts_list = extract_facts_with_comments("data/triples_clean.ttl")

# for each triple add paraphrases
for row in facts_list:
    row["paraphrases"] = make_paraphrases(row)

# save the facts in a jsonl file
facts_jsonl_path = "data/facts.jsonl"
with open(facts_jsonl_path, "w", encoding="utf-8") as f:
    for row in facts_list:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Facts with paraphrases per page saved to {facts_jsonl_path}")



Facts with paraphrases per page saved to data/facts.jsonl


## Point 7

- Generate instruction-response pairs grounded strictly in facts/KG.
- Types of instructions:
    - Factoid QA: “What nutrient is high in X?” → grounded answer + citation (page/section).
    - List/Compare: “List ingredients rich in fiber under 100 kcal/100g.”
    - Reasoning: “If a recipe requires Y technique, which safety temperature applies?”
    - Constraint queries: “Give two vegan recipes under 400 kcal.”
- Include input grounding: attach the top-k supporting facts/triples to each pair during generation (kept for
training or for eval only).
- Include in each jsonl row the attribute "question_type" to recognize the type of instruction, "quantity" for constraint and list types if it asks a certain amount of answers and "constraint_type" for the type of constraint (e.g. "more than two", "less than 200 calories"...).
• Train/val/test split (80/10/10). Ensure no leakage (e.g., split by section/page blocks).

In [627]:
# load facts and create a DataFrame for analysis

facts = []
with open("data/facts.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        facts.append(json.loads(line))

facts_df = pd.DataFrame(facts)

In [628]:
# return top-k facts supporting a query
def get_grounding(subject=None, predicate=None, object_=None, top_k=3):
    df = facts_df.copy()
    if subject: df = df[df['subject'].str.contains(subject, case=False)]
    if predicate: df = df[df['predicate'].str.contains(predicate, case=False)]
    if object_: df = df[df['object'].str.contains(object_, case=False)]
    df = df.head(top_k)
    return df[['subject','predicate','object','page']].to_dict(orient='records')


In [629]:
# define dictionary mapping number words to integers
# this is useful for the "quantity" attribute in facts

NUMBER_WORDS = {
    "zero": 0,
    "one": 1,
    "two": 2,
    "three": 3,
    "four": 4,
    "five": 5,
    "six": 6,
    "seven": 7,
    "eight": 8,
    "nine": 9,
    "ten": 10,
    "eleven": 11,
    "twelve": 12,
    "thirteen": 13,
    "fourteen": 14,
    "fifteen": 15,
    "sixteen": 16,
    "seventeen": 17,
    "eighteen": 18,
    "nineteen": 19,
    "twenty": 20,
    # extend as needed
}


In [630]:
# generate diverse QA pairs from facts
import random

# establish paths
facts_path = Path("data/facts.jsonl")
output_dir = Path("data/train/test/val")
output_dir.mkdir(parents=True, exist_ok=True)

# load facts
facts = []
with open(facts_path, "r", encoding="utf-8") as f:
    for line in f:
        facts.append(json.loads(line))
facts_df = pd.DataFrame(facts)

print(f"Loaded {len(facts_df)} facts from {facts_path}")



# --- helper functions ---
# parse quantity constraints from instruction
def parse_constraint(instruction):
    instruction_lower = instruction.lower()
    quantity = ""
    constraint_type = ""

    # find numerical digits
    numbers = re.findall(r'\b\d+\b', instruction_lower)
    
    # find number words if they are not digits
    if not numbers:
        for word, num in NUMBER_WORDS.items():
            if re.search(rf'\b{word}\b', instruction_lower):
                numbers = [str(num)]
                break

    number = numbers[0] if numbers else ""
    if number:
        quantity = number
        if "over" in instruction_lower:
            constraint_type = "quantity_upper"
        elif "under" in instruction_lower:
            constraint_type = "quantity_lower"
        elif any(x in instruction_lower for x in ["at least", "more", "minimum"]):
            constraint_type = "more"
        elif any(x in instruction_lower for x in ["at most", "less", "maximum"]):
            constraint_type = "less"
        else:
            constraint_type = "exact"

    return constraint_type, quantity


# format page references to get correct page number
def get_page_ref(pages):
    if not pages:
        return ""
    pages = [str(p) for p in pages if p]
    return f"(Source: p.{', p.'.join(pages)})" if pages else ""

# extract grounding facts
def get_grounding(subset):
    grounding = []
    for _, row in subset.iterrows():
        grounding.append({
            "s": row["subject"],
            "p": row["predicate"],
            "o": row["object"],
            "page": row["page"]
        })
    return grounding

# format output for list-type answers
def format_output_list(items, pages):
    if not items:
        return ""
    
    # number the items: 1) item1 2) item2 ...
    items_str = " ".join([f"{i+1}) {itm}" for i, itm in enumerate(items)])
    
    # format page references
    page_ref = get_page_ref(pages)
    if page_ref:
        return f"{items_str}. {page_ref}"
    else:
        return f"{items_str}."



# define templates for different types of questions
# useful to generate diverse QA pairs
# this can help in training models to understand various question formats
TEMPLATES = {
    "hasNutrient": {
        "factoid": [
            "What nutrient is found in {s}?",
            "Which nutrient does {s} provide?",
            "What is the main nutrient in {s}?",
            "What does {s} contain that supports health?",
            "{s} is rich in which nutrient?"
        ],
        "list": [
            "List foods rich in {o}.",
            "Which ingredients are good sources of {o}?",
            "Give examples of foods containing {o}.",
            "Name three foods that are high in {o}."
        ],
        "constraint": [
            "Give two ingredients that contain {o}.",
            "Find vegan foods rich in {o}.",
            "List foods with high {o} content under 200 kcal per 100g."
        ]
    },
    "associatedWithOutcome": {
        "factoid": [
            "What health outcome is {s} associated with?",
            "How does consuming {s} affect health?",
            "{s} is linked to which health condition?",
            "What condition may be influenced by {s}?"
        ],
        "list": [
            "List foods associated with reduced risk of {o}.",
            "Which foods promote {o}?",
            "Name foods linked to {o}."
        ]
    },
    "affectsRiskOf": {
        "factoid": [
            "What health outcome does {s} influence?",
            "Which disease risk is affected by {s}?",
            "How does {s} affect the risk of {o}?",
            "What role does {s} play in preventing {o}?"
        ]
    },
    "usesTechnique": {
        "factoid": [
            "What technique is used to prepare {s}?",
            "How is {s} typically cooked or processed?",
            "What preparation method applies to {s}?"
        ]
    },
    "affectsImpactCategory": {
        "factoid": [
            "What environmental impact is influenced by {s}?",
            "Which sustainability impact is affected by {s}?",
            "How does {s} affect environmental footprint?"
        ],
        "reasoning": [
            "If a recipe uses {s}, what environmental impact may increase?",
            "When {s} is used, which environmental category might be affected?"
        ]
    },
    "hasGuideline": {
        "factoid": [
            "What dietary guideline applies to {s}?",
            "What recommendation is given regarding {s}?",
            "What official advice mentions {s}?"
        ]
    },
    "aimsToImprove": {
        "factoid": [
            "What health outcome does {s} aim to improve?",
            "What benefit does {s} target?",
            "Which condition is addressed by {s}?"
        ]
    },
    "guidelineTargetsImpact": {
        "factoid": [
            "What environmental impact does {s} aim to reduce?",
            "Which sustainability metric is targeted by {s}?"
        ]
    }
}

# define function to convert predicate in sentence such that it doesn't remain as is
# but is converted in a more natural verb form
def predicate_to_verb(predicate):
    mapping = {
        "hasNutrient": "contains",
        "usesTechnique": "is prepared using",
        "hasGuideline": "follows the guideline",
        "recommendsTechnique": "recommends",
        "aimsToImprove": "aims to improve",
        "affectsRiskOf": "affects the risk of",
        "associatedWithOutcome": "is associated with",
        "hasEnvironmentalImpact": "has an environmental impact",
        "guidelineTargetsImpact": "targets environmental impact",
        "affectsImpactCategory": "affects the environmental impact category"
    }
    return mapping.get(predicate, predicate)



# generate functions of different type
def generate_factoid_qa(fact):
    # TYPE 1: factoid-style questions
    s, p, o, page = fact["subject"], fact["predicate"], fact["object"], fact["page"]

    templates = TEMPLATES.get(p, {}).get("factoid", [])
    if not templates:
        instruction = f"What is the relationship between {s.replace('_', ' ')} and {o.replace('_', ' ')}?"
        output = f"{s.replace('_', ' ')} {predicate_to_verb(p)} {o.replace('_', ' ')}. {get_page_ref([page])}"
    else:
        instruction = random.choice(templates).format(s=s.replace("_", " "), o=o.replace("_", " "))
        output = f"{s.replace('_', ' ')} {predicate_to_verb(p)} {o.replace('_', ' ')}. {get_page_ref([page])}"

    grounding = [fact.to_dict()]
    return {"instruction": instruction.strip(), "input": "", "output": output.strip(), "grounding": grounding, "question_type": "factoid", "quantity": "", "constraint_type": ""}


def generate_list_compare(df):
    # TYPE 2: list/compare-style questions
    nutrient_rows = df[df["predicate"] == "hasNutrient"]
    if len(nutrient_rows) < 1:
        return None

    nutrient = random.choice(nutrient_rows["object"].unique().tolist())
    subset_df = nutrient_rows[nutrient_rows["object"] == nutrient]
    sample_size = min(4, len(subset_df))
    if sample_size == 0:
        return None

    subset = subset_df.sample(sample_size)
    ingredients = [s.replace('_', ' ') for s in subset["subject"].tolist()]
    pages = [int(p) for p in subset["page"].tolist() if p]

    templates = TEMPLATES["hasNutrient"]["list"]
    instruction = random.choice(templates).format(o=nutrient.replace("_", " "))
    output = format_output_list(ingredients, pages)
    grounding = get_grounding(subset)
    return {"instruction": instruction.strip(), "input": "", "output": output.strip(), "grounding": grounding, "question_type": "list", "quantity": "", "constraint_type": ""}



def generate_reasoning(df):
    # TYPE 3: reasoning-style questions
    tech_rows = df[df["predicate"] == "affectsImpactCategory"]
    if len(tech_rows) == 0:
        return None

    row = tech_rows.sample(1).iloc[0]
    s, o, page = row["subject"], row["object"], row["page"]
    templates = TEMPLATES["affectsImpactCategory"]["reasoning"]
    instruction = random.choice(templates).format(s=s.replace("_", " "), o=o.replace("_", " "))
    output = f"{s.replace('_', ' ')} affects {o.replace('_', ' ')} impact. {get_page_ref([page])}"
    grounding = [row.to_dict()]
    return {"instruction": instruction.strip(), "input": "", "output": output.strip(), "grounding": grounding, "question_type": "reasoning", "quantity": "", "constraint_type": ""}


def generate_constraint_query(df):
    nutrient_rows = df[df["predicate"] == "hasNutrient"]
    if len(nutrient_rows) == 0:
        return None

    nutrient = random.choice(nutrient_rows["object"].unique().tolist())
    subset_df = nutrient_rows[nutrient_rows["object"] == nutrient]
    sample_size = min(3, len(subset_df))
    if sample_size == 0:
        return None

    subset = subset_df.sample(sample_size)
    ingredients = [s.replace('_', ' ') for s in subset["subject"].tolist()]
    pages = [int(p) for p in subset["page"].tolist() if p]

    templates = TEMPLATES["hasNutrient"]["constraint"]
    instruction = random.choice(templates).format(o=nutrient.replace("_", " "))
    output = format_output_list(ingredients, pages)

    constraint_type, quantity = parse_constraint(instruction)

    return {
        "instruction": instruction.strip(),
        "input": "",
        "output": output.strip(),
        "question_type": "constraint",
        "constraint_type": constraint_type,
        "quantity": quantity,
        "grounding": get_grounding(subset)
    }



# create the dataset
dataset = []

# one QA per fact
for _, fact in facts_df.iterrows():
    dataset.append(generate_factoid_qa(fact))

# extra random examples for different types
for _ in range(40):
    q = generate_list_compare(facts_df)
    if q: dataset.append(q)

for _ in range(20):
    q = generate_reasoning(facts_df)
    if q: dataset.append(q)

for _ in range(20):
    q = generate_constraint_query(facts_df)
    if q: dataset.append(q)

dataset = [d for d in dataset if d]
print(f"Generated {len(dataset)} instruction–response pairs.")


# print preview of dataset
for i, item in enumerate(dataset[:3]):
    print(f"Example {i+1}:")
    print("Instruction:", item["instruction"])
    print("Output:", item["output"])
    print("Grounding:", item["grounding"])
    print("Question Type:", item["question_type"])
    print("Quantity:", item["quantity"])
    print("Constraint Type:", item["constraint_type"])
    print()

# split in train, test and validation
# after shuffling 80% train, 10% val, 10% test
random.shuffle(dataset)
n = len(dataset)
train_end = int(0.8 * n)
val_end = int(0.9 * n)

splits = {
    "train": dataset[:train_end],
    "val": dataset[train_end:val_end],
    "test": dataset[val_end:]
}

for split_name, data in splits.items():
    out_path = output_dir / f"{split_name}_instructions.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for record in data:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"Saved {len(data)} {split_name} examples → {out_path}")


Loaded 485 facts from data/facts.jsonl
Generated 545 instruction–response pairs.
Example 1:
Instruction: Which nutrient does fish provide?
Output: fish contains fat. (Source: p.5)
Grounding: [{'subject': 'fish', 'predicate': 'hasNutrient', 'object': 'fat', 'page': '5', 'paraphrases': ['fish contains fat.', 'fat is a nutrient found in fish.']}]
Question Type: factoid
Quantity: 
Constraint Type: 

Example 2:
Instruction: What official advice mentions grains?
Output: grains follows the guideline increase. (Source: p.5)
Grounding: [{'subject': 'grains', 'predicate': 'hasGuideline', 'object': 'increase', 'page': '5', 'paraphrases': ['grains follows the guideline: increase.', 'The dietary guideline increase applies to grains.']}]
Question Type: factoid
Quantity: 
Constraint Type: 

Example 3:
Instruction: What dietary guideline applies to fruits?
Output: fruits follows the guideline increase. (Source: p.5)
Grounding: [{'subject': 'fruits', 'predicate': 'hasGuideline', 'object': 'increase', '